# Comprehension Dataset — Cleaning, Annotation & Augmentation Pipeline

This notebook implements a full data-preparation pipeline for the ADL comprehension dataset (ACME / AADL formalisms). It covers:

1. **Setup** — Google Drive mount and path configuration
2. **Preprocessing** — noise removal, text normalisation, and quality scoring
3. **Annotation** — ADL-type detection and architectural-element classification
4. **Augmentation** — block reconstruction, structural variation, targeted AADL oversampling, and comment paraphrasing
5. **Dataset splitting** — stratified 80 / 10 / 10 train / val / test partition

> **Note:** All intermediate outputs are stored as `.csv` / `.jsonl` files in `OUTPUT_DIR`. Run cells in order.

## 1. Setup

In [ ]:
# Mount Google Drive and configure I/O paths.
from google.colab import drive
import os

drive.mount('/content/drive')

DATASET_PATH = "/content/drive/MyDrive/dataset_original.jsonl"
OUTPUT_DIR   = "/content/drive/MyDrive/dataset_clean"

os.makedirs(OUTPUT_DIR, exist_ok=True)

if os.path.exists(DATASET_PATH):
    print(f"Source file located : {DATASET_PATH}")
    print(f"Output directory    : {OUTPUT_DIR}")
else:
    raise FileNotFoundError(
        f"Dataset not found at {DATASET_PATH}. "
        "Please ensure dataset_original.jsonl is placed under MyDrive."
    )

## 2. Imports

In [ ]:
import pandas as pd
import re
import json
import os
import random
from collections import Counter

## 3. Preprocessing

The preprocessing stage comprises four sequential steps:
loading the raw JSONL dataset, removing noisy samples, normalising whitespace and punctuation, and filtering by a computed quality score.

In [ ]:
# ===========================================================================
# Step 0 — Load raw dataset (JSONL → DataFrame)
# ===========================================================================
def load_dataset(path: str) -> pd.DataFrame:
    """Read a JSONL file and return a DataFrame with columns [code, comments]."""
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))

    df = pd.DataFrame(records)
    df = df.rename(columns={"input": "code", "target": "comments"})
    df = df[["code", "comments"]]
    print(f"Raw dataset loaded: {len(df)} samples")
    return df


# ===========================================================================
# Step 1 — Remove noisy samples
# ===========================================================================
def remove_noise(df: pd.DataFrame) -> pd.DataFrame:
    """Drop malformed, trivial, or duplicate code-comment pairs."""
    n_initial = len(df)

    # Drop missing values and overly short entries
    df = df.dropna(subset=["code", "comments"])
    df = df[df["code"].str.strip().str.len() > 5]
    df = df[df["comments"].str.strip().str.len() > 10]

    # Discard structural noise patterns (lone braces, ellipsis, etc.)
    noise_patterns = [
        r'^\s*[};{]+\s*$',
        r'^\s*[\.…]+\s*$',
        r'^\s*\}\s*[;,]?\s*$',
        r'^\s*\{\s*$',
        r'^\s*\)\s*[;,]?\s*$',
    ]
    for pattern in noise_patterns:
        df = df[~df["code"].str.match(pattern, na=False)]

    # Discard uninformative comment prefixes
    uninformative_comments = [
        "Other elements of the style",
        "End of the style definition",
        "End of component definition",
        "End of the connector definition",
        "End of properties block",
        "End of bindings block",
        "End of input port",
        "End of output port",
    ]
    for prefix in uninformative_comments:
        df = df[~df["comments"].str.strip().str.startswith(prefix, na=False)]

    # Remove samples containing placeholder ellipsis
    df = df[~df["code"].str.contains(r'\.\.\.|…', na=False)]
    df = df.drop_duplicates(subset=["code", "comments"])

    print(f"Noise removal: {n_initial} → {len(df)} samples "
          f"({n_initial - len(df)} removed)")
    return df


# ===========================================================================
# Step 2 — Text normalisation
# ===========================================================================
def normalise(df: pd.DataFrame) -> pd.DataFrame:
    """Normalise whitespace, collapsed double-quotes, and comment capitalisation."""
    for col in ["code", "comments"]:
        df[col] = df[col].str.strip()
        df[col] = df[col].apply(lambda x: re.sub(r'\s+', ' ', str(x)))
        df[col] = df[col].str.replace('""', '"', regex=False)

    # Ensure comments begin with an upper-case letter
    df["comments"] = df["comments"].apply(
        lambda x: x[0].upper() + x[1:] if len(x) > 1 else x
    )
    print(f"Normalisation complete: {len(df)} samples")
    return df


# ===========================================================================
# Step 3 — Quality scoring and filtering
# ===========================================================================
def score_quality(df: pd.DataFrame) -> pd.DataFrame:
    """Assign a heuristic quality score (0–6) to each code-comment pair."""
    informative_verbs = [
        "Define", "Declare", "Specify", "Ensure",
        "Rule", "Invariant", "Set", "Create",
        "Establish", "Represent", "Model"
    ]
    generic_terms = ["end of", "other", "etc", "..."]

    def compute_score(row) -> int:
        score   = 0
        code    = str(row["code"])
        comment = str(row["comments"])
        if len(code) > 20:    score += 2
        if len(comment) > 30: score += 2
        if any(comment.startswith(v) for v in informative_verbs): score += 1
        if re.search(r'\b[A-Z][a-zA-Z]{2,}\b', code):           score += 1
        if re.search(r'\.\.\.|…|<\.\.\.>', code):              score -= 2
        if any(g in comment.lower() for g in generic_terms):      score -= 1
        return max(0, score)

    df["quality_score"] = df.apply(compute_score, axis=1)
    print("Quality score distribution (0–6):")
    print(df["quality_score"].value_counts().sort_index().to_string())
    return df


def filter_by_quality(df: pd.DataFrame, threshold: int = 2) -> pd.DataFrame:
    """Retain only samples whose quality score meets or exceeds `threshold`."""
    n_before = len(df)
    df_filtered = df[df["quality_score"] >= threshold].copy()
    print(f"Quality filter (threshold={threshold}): "
          f"{n_before} → {len(df_filtered)} samples retained")
    return df_filtered

## 4. Annotation

Two annotation functions are applied in sequence:

- **`annotate_adl_type`** — classifies each snippet as `ACME` or `AADL` using keyword scoring.
- **`annotate_element_type`** — assigns one of 16 fine-grained architectural-element categories via a priority-ordered chain of regular expressions.

In [ ]:
# ===========================================================================
# Step 4 — ADL-type annotation (ACME vs. AADL)
# ===========================================================================
def annotate_adl_type(df: pd.DataFrame) -> pd.DataFrame:
    """Classify each snippet as ACME or AADL based on keyword frequency scoring."""
    keywords_aadl = [
        "subcomponent", "thread", "process", "PortType",
        "Dispatch", "DataType", "DataSize", "Active_on",
        "EventList", "TupleList", "Bindings", "i_", "o_",
        "system implementation", "end "
    ]
    keywords_acme = [
        "Family", "Style", "Component Type", "Connector Type",
        "Port Type", "Role Type", "invariant", "declaresType",
        "forall", "self.PORTS", "self.ROLES", "self.COMPONENTS",
        "PlastikMF", "CUML", "SUMLQoS", "ComponentUML",
        "ComponentQoS", "ConnectorQoS", "InterfaceFonc"
    ]

    def detect_adl(code: str) -> str:
        code_lower = str(code).lower()
        score_aadl = sum(1 for k in keywords_aadl if k.lower() in code_lower)
        score_acme = sum(1 for k in keywords_acme if k.lower() in code_lower)
        return "AADL" if score_aadl > score_acme else "ACME"

    df["adl_type"] = df["code"].apply(detect_adl)
    print("ADL-type distribution:")
    print(df["adl_type"].value_counts().to_string())
    return df


# ===========================================================================
# Step 5 — Architectural-element annotation
# Priority-ordered regex chain (most specific patterns first).
# ===========================================================================
def annotate_element_type(df: pd.DataFrame) -> pd.DataFrame:
    """Assign one of 16 architectural-element categories to each snippet."""

    def classify_element(code: str) -> str:
        s = str(code).strip()

        if re.search(r'(?i)(Family\s+\w+|Style\s+\w+\s*[={])', s):
            return "Family/Style"
        elif re.search(r'(?i)(Glue\s*\{|BaseRole\s+\w+|CrosscuttingRole\s+\w+'
                        r'|crosscuttingRole\d*\s+(before|after)\s+\w+)', s):
            return "Glue"
        elif re.search(r'(?i)(Bindings\s*\{|Attachment\s+\w+\.\w+\s+to\s+\w+\.\w+'
                        r'|attachTo|attachedOrBound|bus\s+access\s+\w+\s*->'
                        r'|connections\s+(bus\s+access|\w+:))', s):
            return "Attachment/Binding"
        elif re.search(r'(?i)(Representation\s+\w+\s*=\s*\{'
                        r'|Representation\s*=\s*\{'
                        r'|On\s*\(\w+\.failure|Detach\s+\w+|Remove\s+\w+)', s):
            return "Representation"
        elif re.search(r'(?i)(Invariant\b|Rule\s+\w+|Heuristic\s+\w+'
                        r'|forall|cardinality|exists\s+\w+.*\|'
                        r'|annex\s+Error_Model|Guard_Event\s*=>'
                        r'|Error_Free|FailedVisible)', s):
            return "Rule/Invariant"
        elif re.search(r'(?i)(modes\s+\w+)', s):
            return "Representation"
        elif re.search(r'(?i)(^data\s+(implementation\s+)?\w+)', s):
            return "DataDefinition"
        elif re.search(r'(?i)(subprogram\s+(implementation\s+)?\w+'
                        r'|^subcomponents\s+\w+)', s):
            return "Component"
        elif re.search(r'(?i)(^bus\s+(implementation\s+)?\w+)', s):
            return "Connector"
        elif re.search(r'(?i)(^memory\s+(implementation\s+)?\w+)', s):
            return "DataDefinition"
        elif re.search(r'(?i)(^device\s+\w+|^processor\s+(implementation\s+)?\w+)', s):
            return "Component"
        elif re.search(r'(?i)(^system\s+(implementation\s+)?\w+)', s):
            return "System"
        elif re.search(r'(?i)(Connector\s+(Type\s+)?\w+\s*[:={])', s):
            return "Connector"
        elif re.search(r'(?i)(Component\s*(Type\s*)?\w*\s*[:={]'
                        r'|ComponentType\s*:|component_category\s+\w+)', s):
            return "Component"
        elif re.search(r'(?i)(Port\s+(Type\s+)?\w+|Ports\s*\{'
                        r'|stdin|stdout|stderr|receiveRequest|sendRequest'
                        r'|externalSocket|securityAuthorization'
                        r'|securityManagementIntf)', s):
            return "Port"
        elif re.search(r'(?i)(Role\s+(Type\s+)?\w+|Roles\s*\{'
                        r'|\bcaller\b|\bcallee\b|requestor|grantor'
                        r'|securityManager|\bsource\s*;|\bsink\s*[;}])', s):
            return "Role"
        elif re.search(r'(?i)(Property\s+Type.*enum|Property\s+Type.*Record'
                        r'|Property\s+Type.*Sequence)', s):
            return "TypeDefinition"
        elif re.search(r'(?i)(Property\s+(Type\s+)?\w+|Properties\s*\{'
                        r'|request-rate|requestRate|sourceCode|idempotent'
                        r'|maxConcurrent|synchronous|multithreaded|maxRoles'
                        r'|queueBufferSize|expectedThroughput'
                        r'|\w+\s*:\s*(float|boolean|integer|externalFile)\s*=)', s):
            return "Property"
        elif re.search(r'(?i)(\w+\s*=\s*\w+\s*\([\w\-\.]+\)'
                        r'|rpc\s*\(\w+\.[\w\-]+,\s*\w+\.[\w\-]+\))', s):
            return "Representation"
        elif re.search(r'(?i)(component\s*\(\w+\)\s*\^'
                        r'|connector\s*\(\w+\)\s*\^)', s):
            return "Constraint"
        elif re.search(r'(?i)(QoS|[Ll]atency|[Bb]andwidth|[Tt]hroughput)', s):
            return "QoS/Service"
        elif re.search(r'(?i)(\bevent\b|\bEventPort\b|\bDispatch\b)', s):
            return "Event"
        elif re.search(r'(?i)(\bthread\b|\bprocess\b|\bsubcomponent\b)', s):
            return "Component"
        else:
            return "Other"

    df["element_type"] = df["code"].apply(classify_element)
    print("Architectural-element distribution:")
    print(df["element_type"].value_counts().to_string())
    n_other = len(df[df["element_type"] == "Other"])
    print(f"\nResidual 'Other': {n_other} samples "
          f"({n_other / len(df) * 100:.1f}%)")
    return df

## 5. Run Preprocessing and Annotation Pipeline

Execute all preprocessing steps in order and save the cleaned dataset.

In [ ]:
def run_preprocessing(input_path: str, output_dir: str) -> pd.DataFrame:
    """Execute the full preprocessing and annotation pipeline."""
    print("=" * 60)
    print("PREPROCESSING PIPELINE — ADL Comprehension Dataset")
    print("=" * 60)

    df = load_dataset(input_path)
    df = remove_noise(df)
    df = normalise(df)
    df = annotate_adl_type(df)
    df = annotate_element_type(df)
    df = score_quality(df)
    df = filter_by_quality(df, threshold=2)

    # Export cleaned dataset
    path_csv   = f"{output_dir}/dataset_clean.csv"
    path_jsonl = f"{output_dir}/dataset_clean.jsonl"

    df.to_csv(path_csv, index=False, encoding="utf-8")

    with open(path_jsonl, "w", encoding="utf-8") as f:
        for _, row in df.iterrows():
            record = {
                "input"        : str(row["code"]),
                "target"       : str(row["comments"]),
                "adl_type"     : str(row["adl_type"]),
                "element_type" : str(row["element_type"]),
                "quality"      : int(row["quality_score"])
            }
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

    # Export cleaning report
    report = {
        "total_samples"         : len(df),
        "adl_distribution"      : df["adl_type"].value_counts().to_dict(),
        "element_distribution"  : df["element_type"].value_counts().to_dict(),
        "mean_quality_score"    : round(df["quality_score"].mean(), 2),
        "high_quality_samples"  : len(df[df["quality_score"] >= 4])
    }
    with open(f"{output_dir}/cleaning_report.json", "w", encoding="utf-8") as f:
        json.dump(report, f, indent=2, ensure_ascii=False)

    print(f"\n{'=' * 60}")
    print(f"PREPROCESSING COMPLETE")
    print(f"{'=' * 60}")
    print(f"  Total samples        : {report['total_samples']}")
    print(f"  ADL distribution     : {report['adl_distribution']}")
    print(f"  Mean quality score   : {report['mean_quality_score']} / 6")
    print(f"  High-quality samples : {report['high_quality_samples']}")
    return df


df_clean = run_preprocessing(DATASET_PATH, OUTPUT_DIR)

## 6. Data Augmentation

Three complementary augmentation strategies are applied sequentially:

| Strategy | Description |
|---|---|
| **Block reconstruction** | Groups adjacent single-line samples into multi-line architectural blocks |
| **Structural variation** | Substitutes entity names (components, connectors, ports, roles) with semantically equivalent alternatives |
| **Comment paraphrasing** | Generates lexically distinct formulations of existing comments without altering their meaning |

In [ ]:
# ===========================================================================
# Augmentation Strategy 1 — Block Reconstruction
# Rationale: the raw dataset contains many isolated single-line snippets.
# Reconstructing complete architectural blocks yields more informative
# training samples that better reflect realistic ADL usage.
# ===========================================================================

BLOCK_START_PATTERNS = [
    r'(?i)^(Style|Family)\s+\w+',
    r'(?i)^Component\s+(Type\s+)?\w+',
    r'(?i)^Connector\s+(Type\s+)?\w+',
    r'(?i)^System\s+\w+',
    r'(?i)^Port\s+Type\s+\w+',
    r'(?i)^Role\s+Type\s+\w+',
    r'(?i)^Property\s+[Tt]ype\s+\w+',
]


def is_block_start(code: str) -> bool:
    """Return True if the snippet opens a new top-level architectural construct."""
    return any(re.match(p, code.strip()) for p in BLOCK_START_PATTERNS)


def reconstruct_blocks(df: pd.DataFrame,
                        min_block_size: int = 3) -> list:
    """Group consecutive rows into architectural blocks and return them as a list."""
    blocks        = []
    current_block = []
    in_block      = False

    for _, row in df.iterrows():
        code = str(row["code"]).strip()
        if is_block_start(code):
            if in_block and len(current_block) >= min_block_size:
                blocks.append(current_block.copy())
            current_block = [row]
            in_block      = True
        elif in_block:
            current_block.append(row)

    if in_block and len(current_block) >= min_block_size:
        blocks.append(current_block.copy())

    print(f"Block reconstruction: {len(blocks)} blocks identified")
    return blocks


def block_to_sample(rows: list) -> dict:
    """Convert a list of rows into a single augmented code-comment pair."""
    codes    = [str(r["code"]).strip()     for r in rows]
    comments = [str(r["comments"]).strip() for r in rows]

    input_text = "\n".join(codes)
    primary    = comments[0]
    supporting = [c for c in comments[1:]
                  if len(c) > 25 and not c.lower().startswith("end of")][:3]
    target = (f"{primary}. It includes: {'; '.join(supporting)}."
              if supporting else primary)

    return {
        "input"        : input_text,
        "target"       : target,
        "adl_type"     : str(rows[0]["adl_type"]),
        "element_type" : str(rows[0]["element_type"]),
        "quality_score": 5,
        "source"       : "augmented_block"
    }


blocks = reconstruct_blocks(df_clean)
df_blocks = pd.DataFrame([block_to_sample(b) for b in blocks])
print(f"Block samples by ADL type:")
print(df_blocks["adl_type"].value_counts().to_string())

# Merge with original dataset (rename columns to align schemas)
df_original = df_clean.rename(
    columns={"code": "input", "comments": "target"}
).copy()
df_original["source"] = "original"

df_augmented_v1 = pd.concat(
    [df_original, df_blocks], ignore_index=True
)
df_augmented_v1.to_csv(
    f"{OUTPUT_DIR}/dataset_augmented_v01.csv", index=False, encoding="utf-8"
)
print(f"\nv01 saved: {len(df_augmented_v1)} total samples")

In [ ]:
# ===========================================================================
# Augmentation Strategy 2 — Structural Variation (entity-name substitution)
# Rationale: replacing architectural entity names with semantically equivalent
# alternatives increases lexical diversity while preserving structural meaning.
# ===========================================================================

SUBSTITUTIONS_ACME = {
    # Components
    "BankServer"       : ["PaymentServer",  "AuthServer",     "DataServer"],
    "Client"           : ["Consumer",        "User",           "Requester"],
    "Server"           : ["Provider",        "Processor",      "Handler"],
    "Database"         : ["Repository",      "DataStore",      "Storage"],
    "OSIComp"          : ["NetworkComp",     "LayerComp",      "ProtocolComp"],
    "ComponentUML"     : ["ComponentSysML",  "ComponentArch",  "ComponentBase"],
    "ComponentQoS"     : ["ComponentRT",     "ComponentSafe",  "ComponentAware"],
    "buffer"           : ["cache",           "queue",          "storage"],
    "calculator"       : ["processor",       "engine",         "solver"],
    "diskIO"           : ["fileIO",          "storageIO",      "dataIO"],
    # Connectors
    "conn2Layers"      : ["conn2Nodes",      "conn2Tiers",     "interlayerConn"],
    "AssemblyUML"      : ["AssemblySysML",   "AssemblyArch",   "AssemblyBase"],
    "ConnectorQoS"     : ["ConnectorRT",     "ConnectorSafe",  "ConnectorAware"],
    # Ports
    "ProvidedPort"     : ["OfferedPort",     "ExposedPort",    "ServicePort"],
    "RequiredPort"     : ["NeededPort",      "DependentPort",  "ClientPort"],
    "ProvidedInterface": ["OfferedInterface","ExposedInterface","ServiceInterface"],
    "RequiredInterface": ["NeededInterface", "DependentInterface","ClientInterface"],
    # Roles
    "caller"           : ["sender",          "initiator",      "requestor"],
    "callee"           : ["receiver",        "responder",      "handler"],
    "source"           : ["producer",        "emitter",        "origin"],
    "sink"             : ["consumer",        "collector",      "destination"],
    # Styles
    "PlastikMF"        : ["LayeredMF",       "ComponentMF",    "ArchMF"],
}

SUBSTITUTIONS_AADL = {
    # Components
    "Login_Worker" : ["Auth_Worker",   "Security_Worker", "Access_Worker"],
    "Login_Manager": ["Auth_Manager",  "Access_Manager",  "User_Manager"],
    "Login"        : ["Authentication","Authorization",   "AccessControl"],
    "Menu"         : ["Dashboard",     "Interface",       "Navigator"],
    "Database"     : ["Repository",    "DataStore",       "Storage"],
    "buffer"       : ["cache",         "queue",           "stack"],
    "calculator"   : ["processor",     "engine",          "solver"],
    # Ports
    "i_username"   : ["i_user_id",     "i_login_id",      "i_account"],
    "i_password"   : ["i_secret",      "i_credential",    "i_passkey"],
    "o_flag"       : ["o_status",      "o_result",        "o_signal"],
    # Roles
    "caller"       : ["sender",        "initiator",       "requestor"],
    "callee"       : ["receiver",      "responder",       "handler"],
}


def generate_variations(row: pd.Series, n_variations: int = 2) -> list:
    """Generate lexical variants of a sample by substituting entity names."""
    code    = str(row["input"])
    comment = str(row["target"])
    subs    = (SUBSTITUTIONS_AADL
               if row["adl_type"] == "AADL"
               else SUBSTITUTIONS_ACME)

    applicable = {k: v for k, v in subs.items()
                  if k in code or k in comment}
    if not applicable:
        return []

    variants = []
    for i in range(n_variations):
        new_code, new_comment = code, comment
        for original, replacements in applicable.items():
            replacement = replacements[i % len(replacements)]
            new_code    = new_code.replace(original, replacement)
            new_comment = new_comment.replace(original, replacement)
        if new_code != code:
            variants.append({
                "input"        : new_code,
                "target"       : new_comment,
                "adl_type"     : row["adl_type"],
                "element_type" : row["element_type"],
                "quality_score": int(row["quality_score"]),
                "source"       : "augmented_variation"
            })
    return variants


df_augmented_v1 = pd.read_csv(f"{OUTPUT_DIR}/dataset_augmented_v01.csv")

all_variations = []
for _, row in df_augmented_v1.iterrows():
    all_variations.extend(generate_variations(row, n_variations=2))

df_variations = pd.DataFrame(all_variations).drop_duplicates(
    subset=["input", "target"])
print(f"Structural variations generated: {len(df_variations)}")
print(df_variations["adl_type"].value_counts().to_string())

df_augmented_v2 = pd.concat(
    [df_augmented_v1, df_variations], ignore_index=True
).drop_duplicates(subset=["input", "target"])

df_augmented_v2.to_csv(
    f"{OUTPUT_DIR}/dataset_augmented_v02.csv", index=False, encoding="utf-8"
)
print(f"\nv02 saved: {len(df_augmented_v2)} total samples ")
print(f"  ACME: {len(df_augmented_v2[df_augmented_v2['adl_type']=='ACME'])} | "
      f"AADL: {len(df_augmented_v2[df_augmented_v2['adl_type']=='AADL'])}")

In [ ]:
# ===========================================================================
# Augmentation Strategy 3 — Targeted AADL Oversampling
# Rationale: the AADL subset is significantly smaller than the ACME subset
# in the original corpus. Additional variations are generated specifically
# from AADL samples to improve class balance before fine-tuning.
# ===========================================================================

SUBSTITUTIONS_AADL_EXTENDED = {
    "Login_Worker" : ["Auth_Worker",   "Security_Worker", "Access_Worker"],
    "Login_Manager": ["Auth_Manager",  "Access_Manager",  "User_Manager"],
    "Database"     : ["Repository",    "DataStore",       "Storage"],
    "Menu"         : ["Dashboard",     "Interface",       "Navigator"],
    "buffer"       : ["cache",         "queue",           "stack"],
    "calculator"   : ["processor",     "engine",          "solver"],
    "diskIO"       : ["fileIO",        "storageIO",       "dataIO"],
    "i_username"   : ["i_user_id",     "i_login_id",      "i_account"],
    "i_password"   : ["i_secret",      "i_credential",    "i_passkey"],
    "o_flag"       : ["o_status",      "o_result",        "o_signal"],
    "Periodic"     : ["Sporadic",      "Aperiodic",       "Hybrid"],
    "IN"           : ["INPUT",         "RECEIVE",         "READ"],
    "OUT"          : ["OUTPUT",        "SEND",            "WRITE"],
}


def oversample_aadl(code: str, comment: str,
                     substitutions: dict) -> list:
    """Generate targeted variations for an individual AADL snippet."""
    variants = []
    for original, replacements in substitutions.items():
        if original in code:
            for replacement in replacements[:2]:
                new_code    = code.replace(original, replacement)
                new_comment = comment.replace(original, replacement)
                if new_code != code:
                    variants.append({
                        "input"        : new_code,
                        "target"       : new_comment,
                        "adl_type"     : "AADL",
                        "element_type" : "Property",
                        "quality_score": 4,
                        "source"       : "augmented_aadl"
                    })
    return variants


df_augmented_v2 = pd.read_csv(f"{OUTPUT_DIR}/dataset_augmented_v02.csv")
df_aadl = df_augmented_v2[df_augmented_v2["adl_type"] == "AADL"].copy()
print(f"AADL samples available: {len(df_aadl)}")

new_aadl_samples = []
for _, row in df_aadl.iterrows():
    new_aadl_samples.extend(
        oversample_aadl(str(row["input"]), str(row["target"]),
                         SUBSTITUTIONS_AADL_EXTENDED)
    )

df_new_aadl = pd.DataFrame(new_aadl_samples).drop_duplicates(
    subset=["input", "target"])

# Exclude samples already present in the dataset
existing_inputs = set(df_augmented_v2["input"].tolist())
df_new_aadl = df_new_aadl[
    ~df_new_aadl["input"].isin(existing_inputs)]

df_augmented_v3 = pd.concat(
    [df_augmented_v2, df_new_aadl], ignore_index=True
)
n_acme = len(df_augmented_v3[df_augmented_v3["adl_type"] == "ACME"])
n_aadl = len(df_augmented_v3[df_augmented_v3["adl_type"] == "AADL"])
print(f"v03 | ACME: {n_acme} | AADL: {n_aadl} | Total: {len(df_augmented_v3)}")

df_augmented_v3.to_csv(
    f"{OUTPUT_DIR}/dataset_augmented_v03.csv", index=False, encoding="utf-8"
)
print(f"v03 saved: {len(df_augmented_v3)} total samples")

In [ ]:
# ===========================================================================
# Augmentation Strategy 4 — Comment Paraphrasing
# Rationale: substituting key architectural verbs and nouns in comments
# with synonyms increases surface-form diversity without altering semantics.
# ===========================================================================

PARAPHRASE_MAP = {
    # Declaration verbs
    "Definition of"   : ["Declaration of",    "Specification of",   "Description of"],
    "Declaration of"  : ["Definition of",     "Specification of",   "Introduction of"],
    "Define"          : ["Declare",            "Specify",            "Introduce"],
    "Declare"         : ["Define",             "Specify",            "Establish"],
    "Specifies"       : ["Defines",            "Indicates",          "Determines"],
    "Defines"         : ["Specifies",          "Establishes",        "Declares"],
    # Constraint verbs
    "Ensures that"    : ["Guarantees that",   "Verifies that",      "Enforces that"],
    "Guarantees that" : ["Ensures that",      "Verifies that",      "Confirms that"],
    "Rule checking"   : ["Constraint verifying","Invariant ensuring","Rule ensuring"],
    # Architectural nouns
    "component"       : ["module",             "element",            "unit"],
    "connector"       : ["link",               "connection",         "binding"],
    "port"            : ["interface",          "endpoint",           "access point"],
    "provided"        : ["offered",            "exposed",            "available"],
    "required"        : ["needed",             "requested",          "consumed"],
}


def paraphrase_comment(comment: str,
                        paraphrase_map: dict) -> list:
    """Generate up to one paraphrase of a comment via term substitution."""
    for original, substitutes in paraphrase_map.items():
        if original.lower() in comment.lower():
            if original in comment:
                new_comment = comment.replace(original, substitutes[0], 1)
            else:
                new_comment = re.sub(re.escape(original), substitutes[0],
                                     comment, count=1, flags=re.IGNORECASE)
            if new_comment != comment:
                return [new_comment]
    return []


df_augmented_v3 = pd.read_csv(f"{OUTPUT_DIR}/dataset_augmented_v03.csv")

# Apply paraphrasing to high-quality AADL samples only
df_aadl_hq = df_augmented_v3[
    (df_augmented_v3["adl_type"] == "AADL") &
    (df_augmented_v3["quality_score"] >= 4)
].copy()
print(f"High-quality AADL samples: {len(df_aadl_hq)}")

paraphrase_records = []
for _, row in df_aadl_hq.iterrows():
    for para in paraphrase_comment(str(row["target"]), PARAPHRASE_MAP):
        if para != str(row["target"]):
            paraphrase_records.append({
                "input"        : str(row["input"]),
                "target"       : para,
                "adl_type"     : "AADL",
                "element_type" : str(row["element_type"]),
                "quality_score": 4,
                "source"       : "augmented_paraphrase"
            })

df_paraphrases = pd.DataFrame(paraphrase_records).drop_duplicates(
    subset=["input", "target"])
print(f"Paraphrase samples generated: {len(df_paraphrases)}")

df_augmented_v4 = pd.concat(
    [df_augmented_v3, df_paraphrases], ignore_index=True
).drop_duplicates(subset=["input", "target"])
print(f"\nv04 | Total: {len(df_augmented_v4)} samples")
print(df_augmented_v4["source"].value_counts().to_string())

In [ ]:
# ===========================================================================
# Class rebalancing — equalise ACME and AADL subset sizes
# Rationale: prevents the dominant ACME class from biasing fine-tuning.
# AADL samples are sorted by provenance quality before truncation.
# ===========================================================================

QUALITY_COL  = "quality_score"
SOURCE_ORDER = {
    "original"             : 0,
    "augmented_aadl"       : 1,
    "augmented_block"      : 2,
    "augmented_variation"  : 3,
    "augmented_paraphrase" : 4,
}

df_acme = df_augmented_v4[df_augmented_v4["adl_type"] == "ACME"]
df_aadl = df_augmented_v4[df_augmented_v4["adl_type"] == "AADL"].copy()

df_aadl["_order"] = df_aadl["source"].map(SOURCE_ORDER).fillna(5)
df_aadl_sorted    = df_aadl.sort_values(
    by=["_order", QUALITY_COL], ascending=[True, False]
).drop(columns=["_order"])
df_aadl_balanced = df_aadl_sorted.iloc[:len(df_acme)]

df_final = pd.concat([df_acme, df_aadl_balanced]).sample(
    frac=1, random_state=42).reset_index(drop=True)

print(f"Rebalanced dataset:")
print(f"  ACME  : {len(df_final[df_final['adl_type']=='ACME'])}")
print(f"  AADL  : {len(df_final[df_final['adl_type']=='AADL'])}")
print(f"  Total : {len(df_final)}")

df_final.to_csv(
    f"{OUTPUT_DIR}/dataset_Comprehension.csv", index=False, encoding="utf-8"
)

# Export JSONL for fine-tuning
with open(f"{OUTPUT_DIR}/dataset_Comprehension.jsonl", "w",
          encoding="utf-8") as f:
    for _, row in df_final.iterrows():
        record = {
            "input"        : str(row["input"]),
            "target"       : str(row["target"]),
            "adl_type"     : str(row["adl_type"]),
            "element_type" : str(row["element_type"]),
            "quality"      : int(row[QUALITY_COL]),
            "source"       : str(row["source"])
        }
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print("dataset_Comprehension.csv and .jsonl saved.")

## 7. Dataset Splitting

The final dataset is partitioned into train / validation / test subsets using a stratified 80 / 10 / 10 split. Stratification is performed independently for each ADL type to preserve class balance across all three splits.

In [ ]:
# ===========================================================================
# Stratified train / val / test split
# ===========================================================================

random.seed(42)

df = pd.read_csv(f"{OUTPUT_DIR}/dataset_Comprehension.csv", encoding="utf-8")
print(f"Dataset loaded: {len(df)} samples")
print(f"  ACME: {len(df[df['adl_type']=='ACME'])} | "
      f"AADL: {len(df[df['adl_type']=='AADL'])}")


def stratified_split(df: pd.DataFrame,
                      train_ratio: float = 0.80,
                      val_ratio:   float = 0.10,
                      test_ratio:  float = 0.10,
                      seed: int = 42):
    """Split a DataFrame into train / val / test with stratification by adl_type."""
    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-9, \
        "Ratios must sum to 1.0"

    def _split(data: pd.DataFrame, tr: float, vr: float):
        data = data.sample(frac=1, random_state=seed).reset_index(drop=True)
        n  = len(data)
        n_train = int(n * tr)
        n_val   = int(n * vr)
        return (data.iloc[:n_train],
                data.iloc[n_train:n_train + n_val],
                data.iloc[n_train + n_val:])

    acme_train, acme_val, acme_test = _split(
        df[df["adl_type"] == "ACME"], train_ratio, val_ratio)
    aadl_train, aadl_val, aadl_test = _split(
        df[df["adl_type"] == "AADL"], train_ratio, val_ratio)

    train = pd.concat([acme_train, aadl_train]).sample(
        frac=1, random_state=seed).reset_index(drop=True)
    val   = pd.concat([acme_val,   aadl_val  ]).sample(
        frac=1, random_state=seed).reset_index(drop=True)
    test  = pd.concat([acme_test,  aadl_test ]).sample(
        frac=1, random_state=seed).reset_index(drop=True)
    return train, val, test


def save_split(df_split: pd.DataFrame, name: str, out_dir: str):
    """Save a split as both CSV and JSONL."""
    os.makedirs(out_dir, exist_ok=True)
    df_split.to_csv(f"{out_dir}/{name}.csv", index=False, encoding="utf-8")

    with open(f"{out_dir}/{name}.jsonl", "w", encoding="utf-8") as f:
        for _, row in df_split.iterrows():
            record = {
                "input"        : str(row["input"]),
                "target"       : str(row["target"]),
                "adl_type"     : str(row["adl_type"]),
                "element_type" : str(row["element_type"]),
                "quality"      : int(row.get("quality_score", 0)),
                "source"       : str(row.get("source", "unknown"))
            }
            f.write(json.dumps(record, ensure_ascii=False) + "\n")


train, val, test = stratified_split(df)

save_split(train, "train_comprehension", OUTPUT_DIR)
save_split(val,   "val_comprehension",   OUTPUT_DIR)
save_split(test,  "test_comprehension",  OUTPUT_DIR)

print("\nSplit summary:")
print(f"  Train : {len(train)} samples")
print(f"  Val   : {len(val)} samples")
print(f"  Test  : {len(test)} samples")
print(f"  Total : {len(train) + len(val) + len(test)} samples")
print(f"\nFiles saved to: {OUTPUT_DIR}")